### Automated Test Case Generation using LangChain 
- Step 1: Define requirements and Schema : This will be strucure of your test case 
- Step 2: Install dependencies 
- Step 3: Setup LLM in LangChain 
- Step 4: Create prompt template 
- Step 5: Create LangChain Prompt & Chain 
- Step 6: Generate Test Cases
- Step 7: Add Validation Layer
- Step 8: Integrate into CI/CD

### Architecture:
User Stories (Jira) ==> LangChain LLM Chain ==> Generate Test Cases (JSON) ==> Validation Layer ==> Automation Script Generation (optional) ==> CI/CD Pipeline

### Objective:
“This is a LangChain-based automated test case generation pipeline where user stories are converted into structured JSON test cases. The pipeline includes a validation layer to ensure schema compliance, automation script generation, and integration into CI/CD. This reduced manual test creation by ~50% and ensured comprehensive coverage including functional, negative, and edge cases.”

### Step 1: Define requirements and Schema : Define the strucure of your test case. Defining a clear schema ensures the generated test cases are structured and can be directly used in automation process
#### Functional ==> Normal workflow 
#### Negative ==> Invalid inputs / error handling 
#### Edge ==> Boundary or special cases 


In [ ]:
# Define JSON structure
{
  "test_case_id": "string",
  "description": "string",
  "type": "functional | negative | edge",
  "steps": ["step1", "step2", "..."],
  "expected_result": "string"
}

#### Step 2: Install dependencies 
- uv init 
- uv add -r requirements.txt (langchain,openai)

### Step 3: Setup LLM in LangChain

In [ ]:
from langchain_community.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
#from langchain.chains import LLMChain

# Initialize OpenAI / Azure LLM
llm = ChatOpenAI(
    model_name="gpt-4", 
    temperature=0.7,
    #openai_api_key=""
    )

### Step 4: Create prompt template 

In [18]:
prompt_template="""
You are a QA engineer. Generate test cases in JSON format from the following story.
Constraints:
- Include functional, negative, and edge cases
-Follow this JSON schema:
{{
  "test_case_id": "",
  "description": "",
  "type": "",
  "steps": [],
  "expected_result": ""
}}
- Output must be valid JSON only.

User Story:
{user_story}
""" 

### Step 5: Create LangChain Prompt & Chain

In [ ]:
chat_prompt = ChatPromptTemplate.from_template(prompt_template)

test_case_chain = LLMChain(
    llm=llm,
    prompt=chat_prompt
)

### Step 6: Generate Test Cases

In [ ]:
user_story="As a user, I want to reset my password via email."

#calling the chain 
response=test_case_chain.run(user_story=user_story)

print(response)

# Expected output 
[
    {
        "test_case_id":"TC001",
        "decription":"Reset password with valid email",
        "type":"functional",
        "steps":["Go to login page", "Click 'Forgot password'","Enter valid email","Submit"],
        "expected_result":"Receive password reset email"
    },
    {
        "test_case_id":"TC002",
        "decription":"Reset password with invalid email",
        "type":"negative",
        "steps":["Go to login page", "Click 'Forgot password'","Enter invalid email","Submit"],
        "expected_result":"Error message displayed"
    }

]

### Step 7: Add Validation Layer

In [ ]:
# Parse JSON and eensure schema compliance

import json
from jsonschema import validate, ValidationError

schema = {
    "type": "object",
    "properties": {
        "test_case_id": {"type": "string"},
        "description": {"type": "string"},
        "type": {"type": "string"},
        "steps": {"type": "array"},
        "expected_result": {"type": "string"}
    },
    "required": ["test_case_id", "description", "type", "steps", "expected_result"]
}

# Validate each test case
test_cases = json.loads(response)
for tc in test_cases:
    try:
        validate(tc, schema)
    except ValidationError as e:
        print("Validation failed:", e)

### Step 8: Integrate into CI/CD
- LangChain chain automate test case generator 
- Trigger on new requirement commits into repo
- Feed directly into automatin pipelines 

#### This allows fully automated generation of structured test cases,m reducing QA cycle time by 50%

### Step 9: Agentic AI Extension : Use LangChain Agents to make it autonomous - to fully autonomous QE pipeline

#### Agent workflow:
1. Requirement Agent ==> Reads new stories 
2. Generation Agent ==> Calls LLM chain to generate test cases 
3. Validation Agent ==> Checks schema 
4. Storage Agent ==> Pushes test cases to test management tool (JIRA, HP ALM system)

